<a href="https://colab.research.google.com/github/NismaK/ML_Homeworkbook/blob/main/ML_zoomcamp_homework_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW06: Trees


In [1]:
import urllib.request
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import xgboost as xgb

url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'
urllib.request.urlretrieve(url, 'car_fuel_efficiency_2026.csv')

df = pd.read_csv('car_fuel_efficiency_2026.csv')
df = df.fillna(0)

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

target = 'fuel_efficiency_mpg'
y_train = df_train[target].values
y_val = df_val[target].values

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(df_train.drop(columns=[target]).to_dict(orient='records'))
X_val = dv.transform(df_val.drop(columns=[target]).to_dict(orient='records'))

def rmse(y, p): return float(np.sqrt(mean_squared_error(y, p)))
print(df.shape, X_train.shape, X_val.shape)

(10000, 11) (6000, 16) (2000, 16)


## Q1

In [2]:
dt = DecisionTreeRegressor(max_depth=1, random_state=1)
dt.fit(X_train, y_train)

feature = dv.get_feature_names_out()[dt.tree_.feature[0]]
original_feature = feature.split('=')[0]
print('Split feature:', feature)
print('Q1 answer:', original_feature)

Split feature: model_year
Q1 answer: model_year


## Q2

In [3]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
q2 = rmse(y_val, rf.predict(X_val))
print('Q2 RMSE:', round(q2, 4))
print('Q2 answer:', min([0.1837, 1.837, 18.37, 183.7], key=lambda x: abs(x-q2)))

Q2 RMSE: 1.8371
Q2 answer: 1.837


## Q3

In [4]:
scores = {}
for n in [10, 50, 100, 150]:
    model = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    model.fit(X_train, y_train)
    scores[n] = round(rmse(y_val, model.predict(X_val)), 3)

print(scores)
q3 = min(scores, key=scores.get)
print('Q3 answer:', q3)

{10: 1.837, 50: 1.772, 100: 1.768, 150: 1.769}
Q3 answer: 100


## Q4

In [5]:
mean_scores = {}
for depth in [10, 15, 20, 25]:
    vals = []
    for n in [10, 50, 100, 150]:
        model = RandomForestRegressor(n_estimators=n, max_depth=depth, random_state=1, n_jobs=-1)
        model.fit(X_train, y_train)
        vals.append(rmse(y_val, model.predict(X_val)))
    mean_scores[depth] = round(float(np.mean(vals)), 3)

print(mean_scores)
q4 = min(mean_scores, key=mean_scores.get)
print('Q4 answer:', q4)

{10: 1.755, 15: 1.784, 20: 1.788, 25: 1.786}
Q4 answer: 10


## Q5

In [6]:
rf_imp = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf_imp.fit(X_train, y_train)

importances = pd.Series(rf_imp.feature_importances_, index=dv.get_feature_names_out())
importances.index = [x.split('=')[0] for x in importances.index]
importances = importances.groupby(level=0).sum().sort_values(ascending=False)
print(importances.head())
q5 = importances[['vehicle_weight','horsepower','acceleration','engine_displacement']].idxmax()
print('Q5 answer:', q5)

model_year             0.296001
vehicle_weight         0.203879
fuel_type              0.193551
horsepower             0.078017
engine_displacement    0.061005
dtype: float64
Q5 answer: vehicle_weight


## Q6

In [7]:
features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)
watchlist = [(dtrain, 'train'), (dval, 'val')]

xgb_scores = {}
for eta in [0.3, 0.1]:
    params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,
        'objective': 'reg:squarederror',
        'nthread': 8,
        'seed': 1,
        'verbosity': 1,
    }
    model = xgb.train(params, dtrain, num_boost_round=100, evals=watchlist, verbose_eval=False)
    xgb_scores[eta] = round(rmse(y_val, model.predict(dval)), 4)

print(xgb_scores)
q6 = min(xgb_scores, key=xgb_scores.get)
print('Q6 answer:', q6)

{0.3: 1.8265, 0.1: 1.7248}
Q6 answer: 0.1


## Final answers

In [8]:
print({
    'Q1': original_feature,
    'Q2': round(q2, 4),
    'Q3': q3,
    'Q4': q4,
    'Q5': q5,
    'Q6': q6,
})

{'Q1': 'model_year', 'Q2': 1.8371, 'Q3': 100, 'Q4': 10, 'Q5': 'vehicle_weight', 'Q6': 0.1}
